# Notebook 15: Energy Cascades & Hamiltonian Decomposition
**Understanding Energy Flow and Conservative vs Dissipative Dynamics**

## What You'll Learn

In this notebook, you'll explore how energy flows through neural networks:

- **Energy Cascades**: Hierarchical energy flow from input to output
- **Dissipation Analysis**: Where and how energy is dissipated
- **Spectral Analysis**: Power law exponents in layer activations
- **Hamiltonian Decomposition**: Separating conservative and dissipative dynamics
- **Phase Space Geometry**: Understanding state space structure

**Prerequisites**: 
- Notebooks 01 (Intro), 12 (Thermodynamics), 14 (Neural ODEs)
- Basic understanding of energy conservation
- Familiarity with phase space concepts

**Time Estimate**: 75-90 minutes

---

## Background: Energy in Neural Networks

### Two Perspectives on Energy

**1. Computational Energy** (cascades through layers):
- Input layer "injects" energy (activation variance)
- Each layer transforms and dissipates energy
- Output layer contains remaining energy
- Total: Input = Output + Dissipation

**2. Dynamical Energy** (Hamiltonian mechanics):
- Conservative part: Energy-preserving transformations
- Dissipative part: Energy-dissipating operations
- Total dynamics = Conservative + Dissipative

### The Turbulent Cascade Analogy

Neural network energy flow resembles **turbulent energy cascades** in fluid dynamics:

**Richardson Cascade** (1922): "Big whorls have little whorls, which feed on their velocity..."

**Kolmogorov Theory** (1941): Energy cascade follows power law
$$E(k) \propto k^{-5/3}$$

In neural networks:
- Early layers = large-scale structure
- Deep layers = fine-scale features
- Energy cascades from coarse to fine

### Key Papers

- **Richardson (1922)**: "Weather Prediction by Numerical Process"
- **Kolmogorov (1941)**: "The local structure of turbulence"
- **Goldstein et al. (2002)**: "Classical Mechanics" (Hamiltonian formalism)
- **Saxe et al. (2014)**: "Exact solutions to the nonlinear dynamics of learning"

---

In [ ]:
# Imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import neuros-mechint energy analyzers
from neuros_mechint.energy_flow import (
    EnergyCascadeAnalyzer,
    HamiltonianDecomposer
)

print("✓ All imports successful!")

---

## Part 1: Energy Cascades Through Network Layers

### The Cascade Concept

**Energy Flow**:
```
Input → Layer 1 → Layer 2 → ... → Layer N → Output
   E₀      ↓ ε₁     ↓ ε₂          ↓ εₙ       Eₙ
```

Where $\epsilon_i$ is energy dissipated at layer $i$.

**Energy Conservation**:
$$E_0 = E_N + \sum_{i=1}^{N} \epsilon_i$$

### Energy Metrics

Multiple ways to define "energy":

**1. Variance** (statistical energy):
$$E = \text{Var}(x) = \frac{1}{n}\sum_i (x_i - \bar{x})^2$$

**2. L2 Norm** (geometric energy):
$$E = \|x\|_2^2 = \sum_i x_i^2$$

**3. Frobenius Norm** (matrix energy):
$$E = \|X\|_F^2 = \sum_{i,j} X_{ij}^2$$

We'll use **variance** as it's scale-invariant.

---

In [ ]:
# Create feedforward network for cascade analysis
class DeepNetwork(nn.Module):
    """Deep network to observe energy cascade."""
    def __init__(self, input_size=784, hidden_sizes=[256, 128, 64, 32], num_classes=10):
        super().__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Initialize model
model = DeepNetwork(input_size=784, hidden_sizes=[256, 128, 64, 32], num_classes=10)
model = model.to(device)

print(f"Deep Network:")
print(f"  - Input size: 784")
print(f"  - Hidden layers: {[256, 128, 64, 32]}")
print(f"  - Total layers: {len([m for m in model.modules() if isinstance(m, nn.Linear)])}")
print(f"  - Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Initialize Energy Cascade Analyzer
cascade = EnergyCascadeAnalyzer(
    model=model,
    energy_metric='variance',  # Use variance as energy
    track_spectrum=True,       # Compute spectral analysis
    verbose=True
)

print("EnergyCascadeAnalyzer initialized:")
print(f"  - Energy metric: {cascade.energy_metric}")
print(f"  - Spectral analysis: {cascade.track_spectrum}")

In [ ]:
# Create input data
batch_size = 32
inputs = torch.randn(batch_size, 784).to(device)

print(f"Input data: {inputs.shape}")
print(f"Input energy: {inputs.var().item():.4f}")
print(f"\nAnalyzing energy cascade...\n")

# Analyze cascade
result_cascade = cascade.analyze_cascade(inputs=inputs)

print("\n✓ Energy cascade analysis complete!")
print(f"\nGlobal Metrics:")
print(f"  - Total input energy: {result_cascade.total_input_energy:.4f}")
print(f"  - Total output energy: {result_cascade.total_output_energy:.4f}")
print(f"  - Total dissipation: {result_cascade.total_dissipation:.4f}")
print(f"  - Energy balance: {result_cascade.energy_balance:.6f} (should be ≈ 0)")
print(f"  - Cascade exponent: {result_cascade.cascade_exponent:.3f}" if result_cascade.cascade_exponent else "  - Cascade exponent: N/A")

In [ ]:
# Examine per-layer energetics
print("\nPer-Layer Energy Analysis:\n")
print("="*80)

for i, layer_e in enumerate(result_cascade.layer_energetics):
    print(f"\nLayer {i}: {layer_e.layer_name}")
    print(f"  Input Energy:         {layer_e.input_energy:.6f}")
    print(f"  Output Energy:        {layer_e.output_energy:.6f}")
    print(f"  Dissipation:          {layer_e.dissipation:.6f}")
    print(f"  Transfer Efficiency:  {layer_e.transfer_efficiency:.3f} ({layer_e.transfer_efficiency*100:.1f}%)")
    print(f"  Spectral Entropy:     {layer_e.spectral_entropy:.3f}")
    print(f"  Sparsity:             {layer_e.sparsity:.3f}")
    
print("\n" + "="*80)

In [ ]:
# Visualize energy cascade
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

layer_names = [le.layer_name.split('.')[-1] if '.' in le.layer_name else le.layer_name 
               for le in result_cascade.layer_energetics]
layer_indices = list(range(len(layer_names)))

# Plot 1: Energy flow
input_energies = [le.input_energy for le in result_cascade.layer_energetics]
output_energies = [le.output_energy for le in result_cascade.layer_energetics]
dissipations = [le.dissipation for le in result_cascade.layer_energetics]

axes[0, 0].plot(layer_indices, input_energies, 'o-', label='Input Energy', 
               linewidth=2, markersize=6, color='green', alpha=0.7)
axes[0, 0].plot(layer_indices, output_energies, 's-', label='Output Energy', 
               linewidth=2, markersize=6, color='blue', alpha=0.7)
axes[0, 0].bar(layer_indices, dissipations, alpha=0.5, color='red', 
              width=0.4, label='Dissipation')
axes[0, 0].set_xlabel('Layer Index')
axes[0, 0].set_ylabel('Energy')
axes[0, 0].set_title('Energy Flow Through Layers', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Transfer efficiency
transfer_eff = [le.transfer_efficiency for le in result_cascade.layer_energetics]
axes[0, 1].plot(layer_indices, transfer_eff, 'o-', linewidth=2, 
               markersize=8, color='purple', alpha=0.7)
axes[0, 1].axhline(y=1.0, linestyle='--', color='gray', alpha=0.5, label='Perfect transfer')
axes[0, 1].set_xlabel('Layer Index')
axes[0, 1].set_ylabel('Transfer Efficiency')
axes[0, 1].set_title('Energy Transfer Efficiency', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Dissipation rate
if len(result_cascade.dissipation_rate) > 0:
    axes[1, 0].plot(layer_indices, result_cascade.dissipation_rate, 'o-', 
                   linewidth=2, markersize=8, color='darkred', alpha=0.7)
    axes[1, 0].set_xlabel('Layer Index')
    axes[1, 0].set_ylabel('Dissipation Rate')
    axes[1, 0].set_title('Energy Dissipation Rate', fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Spectral entropy
spectral_entropies = [le.spectral_entropy for le in result_cascade.layer_energetics]
axes[1, 1].bar(layer_indices, spectral_entropies, alpha=0.7, color='teal')
axes[1, 1].set_xlabel('Layer Index')
axes[1, 1].set_ylabel('Spectral Entropy')
axes[1, 1].set_title('Spectral Entropy per Layer', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Energy decreases through layers (dissipation)")
print("  - ReLU layers cause most dissipation (zeroing negatives)")
print("  - Transfer efficiency < 1 indicates information loss")
print("  - Spectral entropy reveals complexity of activations")

### Energy Conservation Check

**Fundamental Law**: Energy must be conserved

$$E_{\text{in}} = E_{\text{out}} + \sum_{i} \epsilon_i$$

Let's verify:

---

In [ ]:
# Energy conservation verification
print("Energy Conservation Check:\n")
print("="*60)

E_in = result_cascade.total_input_energy
E_out = result_cascade.total_output_energy
E_dissipated = result_cascade.total_dissipation
E_balance = result_cascade.energy_balance

print(f"Input Energy:        {E_in:.6f}")
print(f"Output Energy:       {E_out:.6f}")
print(f"Total Dissipation:   {E_dissipated:.6f}")
print(f"-" * 60)
print(f"Sum (Out + Diss):    {E_out + E_dissipated:.6f}")
print(f"Balance (In - Sum):  {E_balance:.6f}")
print(f"="*60)

# Check if conserved (within numerical precision)
is_conserved = abs(E_balance) < 0.01 * E_in

if is_conserved:
    print("\n✓ Energy is conserved (within numerical precision)")
else:
    print(f"\n⚠ Energy balance issue: {E_balance:.6f}")
    print("  This may be due to numerical precision or complex nonlinearities")

---

## Part 2: Hamiltonian Decomposition

### Conservative vs Dissipative Dynamics

Any dynamical system can be decomposed into:

$$\frac{dx}{dt} = F_{\text{conservative}}(x) + F_{\text{dissipative}}(x)$$

**Conservative Part** (Hamiltonian):
- Preserves energy and phase space volume
- Reversible in time
- Divergence-free: $\nabla \cdot F_c = 0$
- Example: Rotations, oscillations

**Dissipative Part**:
- Dissipates energy
- Irreversible
- Contracts phase space: $\nabla \cdot F_d < 0$
- Example: Damping, friction

### Helmholtz Decomposition

**Theorem**: Any vector field can be uniquely decomposed as:

$$F = \nabla \phi + \nabla \times A$$

Where:
- $\nabla \phi$: Gradient (curl-free, conservative)
- $\nabla \times A$: Curl (divergence-free, conservative)
- Remainder: Dissipative

---

In [ ]:
# Create simple dynamical system for Hamiltonian analysis
class MixedDynamics(nn.Module):
    """
    Dynamical system with both conservative and dissipative parts.
    
    Conservative: Rotation (preserves energy)
    Dissipative: Damping (dissipates energy)
    """
    def __init__(self, rotation_strength=0.5, damping=0.1):
        super().__init__()
        self.rotation_strength = rotation_strength
        self.damping = damping
        
    def forward(self, state):
        # state = [x, y] (2D for visualization)
        x = state[:, 0:1]
        y = state[:, 1:2]
        
        # Conservative part: rotation
        dx_conservative = self.rotation_strength * y
        dy_conservative = -self.rotation_strength * x
        
        # Dissipative part: damping
        dx_dissipative = -self.damping * x
        dy_dissipative = -self.damping * y
        
        # Total dynamics
        dx = dx_conservative + dx_dissipative
        dy = dy_conservative + dy_dissipative
        
        return torch.cat([dx, dy], dim=1)

# Create system
dynamics = MixedDynamics(rotation_strength=0.5, damping=0.1)

print("Mixed Dynamical System:")
print(f"  - Rotation strength: {dynamics.rotation_strength}")
print(f"  - Damping coefficient: {dynamics.damping}")
print(f"  - State dimension: 2D (x, y)")

In [ ]:
# Initialize Hamiltonian Decomposer
decomposer = HamiltonianDecomposer(
    model=dynamics,
    dt=0.01,
    method='helmholtz',
    verbose=True
)

print("HamiltonianDecomposer initialized:")
print(f"  - Time step dt: {decomposer.dt}")
print(f"  - Method: {decomposer.method}")

In [ ]:
# Decompose dynamics
initial_states = torch.tensor([[1.0, 0.0]], dtype=torch.float32)

print(f"Initial state: {initial_states[0].tolist()}")
print(f"\nDecomposing dynamics into conservative and dissipative parts...\n")

result_hamiltonian = decomposer.decompose_dynamics(
    initial_states=initial_states,
    n_timesteps=200,
    compute_lyapunov=False
)

print("\n✓ Hamiltonian decomposition complete!")
print(f"\nDecomposition Results:")
print(f"  - Hamiltonian fraction:  {result_hamiltonian.hamiltonian_fraction:.3f} (conservative)")
print(f"  - Dissipation fraction:  {result_hamiltonian.dissipation_fraction:.3f} (dissipative)")
print(f"  - Sum of fractions:      {result_hamiltonian.hamiltonian_fraction + result_hamiltonian.dissipation_fraction:.3f}")

In [ ]:
# Visualize Hamiltonian decomposition
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

times = result_hamiltonian.times
states = result_hamiltonian.states
conservative = result_hamiltonian.components.conservative_field
dissipative = result_hamiltonian.components.dissipative_field
hamiltonian = result_hamiltonian.hamiltonian_over_time
divergence = result_hamiltonian.components.divergence

# Plot 1: Phase portrait
axes[0, 0].plot(states[:, 0], states[:, 1], linewidth=2, alpha=0.7, color='purple')
axes[0, 0].scatter(states[0, 0], states[0, 1], s=100, c='green', 
                  marker='o', edgecolor='black', zorder=5, label='Start')
axes[0, 0].scatter(states[-1, 0], states[-1, 1], s=100, c='red', 
                  marker='X', edgecolor='black', zorder=5, label='End')
axes[0, 0].scatter(0, 0, s=150, c='gold', marker='*', 
                  edgecolor='black', zorder=5, label='Fixed Point')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].set_title('Phase Portrait (Total Dynamics)', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axis('equal')

# Plot 2: Hamiltonian function
axes[0, 1].plot(times, hamiltonian, linewidth=2, color='blue', alpha=0.8)
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('H(q,p)')
axes[0, 1].set_title('Hamiltonian Function Over Time', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Energy components
conservative_energy = np.linalg.norm(conservative, axis=1)
dissipative_energy = np.linalg.norm(dissipative, axis=1)

axes[0, 2].plot(times, conservative_energy, linewidth=2, 
               label='Conservative', color='green', alpha=0.7)
axes[0, 2].plot(times, dissipative_energy, linewidth=2, 
               label='Dissipative', color='red', alpha=0.7)
axes[0, 2].set_xlabel('Time')
axes[0, 2].set_ylabel('Field Magnitude')
axes[0, 2].set_title('Conservative vs Dissipative Components', fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Divergence
axes[1, 0].plot(times, divergence, linewidth=2, color='purple', alpha=0.8)
axes[1, 0].axhline(y=0, linestyle='--', color='gray', alpha=0.5)
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('∇·F (volume change rate)')
axes[1, 0].set_title('Phase Space Divergence', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Phase space volume
volume = result_hamiltonian.components.phase_space_volume
axes[1, 1].plot(times, volume, linewidth=2, color='teal', alpha=0.8)
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Phase Space Volume')
axes[1, 1].set_title('Volume Evolution (Liouville)', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Dissipation rate
dissipation_rate = result_hamiltonian.dissipation_rate_over_time
axes[1, 2].plot(times, dissipation_rate, linewidth=2, color='darkred', alpha=0.8)
axes[1, 2].set_xlabel('Time')
axes[1, 2].set_ylabel('Dissipation Rate')
axes[1, 2].set_title('Energy Dissipation Rate', fontweight='bold')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Spiral trajectory shows both rotation (conservative) and damping (dissipative)")
print("  - Hamiltonian decreases due to dissipation")
print("  - Divergence < 0 indicates phase space contraction (dissipative)")
print("  - Volume decreases as system approaches fixed point")

### Understanding the Decomposition

**Conservative Component**:
- Causes rotation in phase space
- Would preserve energy if alone
- Reversible dynamics

**Dissipative Component**:
- Causes contraction toward fixed point
- Dissipates energy
- Irreversible dynamics

**Fractions**:
- Hamiltonian fraction ≈ 0.8 → Mostly conservative
- Dissipation fraction ≈ 0.2 → Some dissipation

This matches our system design: strong rotation (0.5), weak damping (0.1)!

---

## Part 3: Connecting Energy Cascades and Hamiltonian Structure

### The Big Picture

**Energy Cascades** tell us:
- How much energy flows through each layer
- Where dissipation occurs
- Efficiency of information transfer

**Hamiltonian Decomposition** tells us:
- How much of the dynamics is reversible vs irreversible
- Structure of the phase space
- Nature of computation (conservative transformations vs lossy operations)

**Together**: Complete thermodynamic and dynamical picture!

### Neural Network Implications

**High Hamiltonian Fraction** (mostly conservative):
- Information-preserving transformations
- Potentially reversible computation
- Lower thermodynamic cost (Landauer)

**High Dissipation Fraction** (mostly dissipative):
- Information loss
- Irreversible computation
- Higher thermodynamic cost

**Optimal Design**: Balance between:
- Conservative (preserve information)
- Dissipative (compress and filter)

---

## Summary & Key Takeaways

### What You Learned

1. **Energy Cascades**:
   - Energy flows hierarchically through network layers
   - Dissipation occurs primarily at nonlinearities (ReLU)
   - Total energy is conserved: $E_{in} = E_{out} + \sum \epsilon_i$
   - Transfer efficiency quantifies information preservation

2. **Hamiltonian Decomposition**:
   - Any dynamics = Conservative + Dissipative
   - Conservative: energy-preserving, reversible
   - Dissipative: energy-dissipating, irreversible
   - Fractions reveal computational structure

3. **Phase Space Analysis**:
   - Divergence measures volume change
   - Negative divergence → contraction → dissipation
   - Phase space volume evolution follows Liouville's theorem

### Key Equations

**Energy Conservation**:
$$E_0 = E_N + \sum_{i=1}^{N} \epsilon_i$$

**Helmholtz Decomposition**:
$$F = F_{\text{conservative}} + F_{\text{dissipative}}$$

**Divergence (Volume Change)**:
$$\frac{dV}{dt} = V \nabla \cdot F$$

**Kolmogorov Cascade**:
$$E(k) \propto k^{-5/3}$$

### Practical Applications

1. **Architecture Design**: Minimize unnecessary dissipation
2. **Efficiency Optimization**: Identify bottleneck layers
3. **Reversible Networks**: Design conservative transformations
4. **Generalization**: Conservative dynamics may generalize better
5. **Thermodynamic Cost**: Connect to Landauer analysis (Notebook 12)

### Next Steps

- **Notebook 16**: Integrate all analyses into automated pipeline
- **Research**: Design networks with optimal energy flow
- **Applications**: Use energy analysis for debugging and optimization

### References

1. **Richardson (1922)**: "Weather Prediction by Numerical Process"
2. **Kolmogorov (1941)**: "The local structure of turbulence in incompressible viscous fluid"
3. **Goldstein et al. (2002)**: "Classical Mechanics" (Hamiltonian formalism)
4. **Arnold (1989)**: "Mathematical Methods of Classical Mechanics"
5. **Saxe et al. (2014)**: "Exact solutions to the nonlinear dynamics of learning"

---

## Exercises

### Exercise 1: Different Activation Functions
Compare energy cascades across different activation functions (ReLU, GELU, Swish).

**Starter code**:

In [ ]:
# Exercise 1: Activation function comparison
# TODO: Create networks with different activations
# TODO: Analyze cascade for each
# TODO: Compare dissipation rates

# Your code here:
pass

### Exercise 2: Reversible Layers
Design a layer that is purely Hamiltonian (no dissipation).

**Starter code**:

In [ ]:
# Exercise 2: Reversible layer design
class ReversibleLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # TODO: Design transformation that preserves energy
        pass
    
    def forward(self, x):
        # TODO: Implement reversible transformation
        pass

# TODO: Test with Hamiltonian decomposer
# Your code here:
pass

### Exercise 3: Cascade Exponent Analysis
Investigate how cascade exponent changes with network depth.

**Starter code**:

In [ ]:
# Exercise 3: Depth vs cascade exponent
depths = [2, 4, 6, 8, 10]
exponents = []

# TODO: Train networks of different depths
# TODO: Measure cascade exponent for each
# TODO: Plot depth vs exponent

# Your code here:
pass

---

**Congratulations!** You've mastered energy cascade analysis and Hamiltonian decomposition. You can now understand both the thermodynamic and dynamical structure of neural networks.

Continue to **Notebook 16: Pipeline & Database Demo** to learn how to automate and integrate all Phase 2 analyses into an efficient workflow.